# LAB 5 — Dashboard Naive RAG vs. Advanced RAG

**MBA em RAG & CAG Aplicados a Direito e Segurança Pública**  
**Aula 5 — Comparação de Pipelines e Análise de Qualidade**

---

## Objetivo

Comparar visualmente as 4 métricas RAGAS entre o **Naive RAG (#T01)** e o **Advanced RAG (#T02)** usando um dashboard Matplotlib completo, identificando ganhos de qualidade obtidos com reranking.

## Técnicas Comparadas

| Tag | Pipeline | Retrieval | Geração |
|-----|----------|-----------|--------|
| #T01 | **Naive RAG** | kNN top-3 (BGE-M3) | vLLM Llama 3.1 8B |
| #T02 | **Advanced RAG** | kNN top-10 + Reranking BGE-Reranker-v2-m3 → top-3 | vLLM Llama 3.1 8B |

## Metas Mínimas de Qualidade (ABNT NBR ISO/IEC 25010)

| Métrica RAGAS | Meta Mínima | Justificativa |
|---------------|-------------|---------------|
| **Faithfulness** | ≥ 0,80 | Respostas devem ser suportadas pelos contextos recuperados |
| **Answer Relevancy** | ≥ 0,75 | Resposta deve ser pertinente à pergunta formulada |
| **Context Recall** | ≥ 0,70 | Contextos devem cobrir as informações do ground truth |
| **Context Precision** | ≥ 0,70 | Contextos recuperados devem ser relevantes (sem ruído) |

---

> **Norma:** ABNT NBR ISO/IEC 25010:2023 — Qualidade de Produto de Software  
> **Ambiente:** Google Colab • Python 3.11+ • GPU T4 • OpenSearch 3.x • vLLM (Llama 3.1 8B Instruct)


## 1. Instalação de Dependências

Instala todos os pacotes necessários para execução no Google Colab.

In [ ]:
# Instalação das dependências — execute apenas uma vez por sessão Colab
!pip install -q \
    ragas==0.1.21 \
    datasets==2.20.0 \
    langchain==0.2.16 \
    langchain-openai==0.1.25 \
    langchain-community==0.2.16 \
    sentence-transformers==3.1.1 \
    opensearch-py==2.7.1 \
    matplotlib==3.9.2 \
    seaborn==0.13.2 \
    pandas==2.2.2 \
    numpy==1.26.4

print("✓ Dependências instaladas com sucesso.")

## 2. Importações e Configuração de Variáveis de Ambiente

In [ ]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from datetime import datetime

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────
# Variáveis de ambiente — ajuste conforme seu ambiente de lab
# ─────────────────────────────────────────────────────────────
os.environ["VLLM_BASE_URL"]       = os.getenv("VLLM_BASE_URL",       "http://localhost:8000/v1")
os.environ["VLLM_MODEL_NAME"]     = os.getenv("VLLM_MODEL_NAME",     "meta-llama/Llama-3.1-8B-Instruct")
os.environ["VLLM_API_KEY"]        = os.getenv("VLLM_API_KEY",        "token-lab-mba-2025")
os.environ["OPENSEARCH_HOST"]     = os.getenv("OPENSEARCH_HOST",     "localhost")
os.environ["OPENSEARCH_PORT"]     = os.getenv("OPENSEARCH_PORT",     "9200")
os.environ["OPENSEARCH_INDEX"]    = os.getenv("OPENSEARCH_INDEX",    "corpus_juridico")
os.environ["LANGFUSE_HOST"]       = os.getenv("LANGFUSE_HOST",       "http://localhost:3000")
os.environ["LANGFUSE_PUBLIC_KEY"] = os.getenv("LANGFUSE_PUBLIC_KEY", "pk-lf-mba-lab5")
os.environ["LANGFUSE_SECRET_KEY"] = os.getenv("LANGFUSE_SECRET_KEY", "sk-lf-mba-lab5")
os.environ["EMBEDDER_MODEL"]      = os.getenv("EMBEDDER_MODEL",      "BAAI/bge-m3")
os.environ["RERANKER_MODEL"]      = os.getenv("RERANKER_MODEL",      "BAAI/bge-reranker-v2-m3")

# Caminhos dos arquivos de entrada
DATASET_PATH  = "dataset_avaliacao_completo.json"
BASELINE_PATH = "ragas_baseline_resultados.csv"

# Metas mínimas RAGAS
METAS = {
    "faithfulness":     0.80,
    "answer_relevancy": 0.75,
    "context_recall":   0.70,
    "context_precision":0.70,
}

print(f"✓ Variáveis de ambiente configuradas.")
print(f"  vLLM:      {os.environ['VLLM_BASE_URL']}")
print(f"  OpenSearch: {os.environ['OPENSEARCH_HOST']}:{os.environ['OPENSEARCH_PORT']}")
print(f"  LangFuse:  {os.environ['LANGFUSE_HOST']}")
print(f"  Embedder:  {os.environ['EMBEDDER_MODEL']}")
print(f"  Reranker:  {os.environ['RERANKER_MODEL']}")

## 3. Health Checks de Conexões

In [ ]:
import requests
from opensearchpy import OpenSearch
from langchain_openai import ChatOpenAI
from sentence_transformers import SentenceTransformer

STATUS = {}

# ── Health Check: vLLM ──────────────────────────────────────
try:
    resp = requests.get(
        f"{os.environ['VLLM_BASE_URL'].rstrip('/v1')}/health",
        timeout=5
    )
    STATUS["vLLM"] = "✅ OK" if resp.status_code == 200 else f"⚠ HTTP {resp.status_code}"
except Exception as e:
    STATUS["vLLM"] = f"❌ Offline ({e})"

# ── Health Check: OpenSearch ────────────────────────────────
try:
    os_client = OpenSearch(
        hosts=[{"host": os.environ["OPENSEARCH_HOST"],
                "port": int(os.environ["OPENSEARCH_PORT"])}],
        http_compress=True,
        use_ssl=False,
        verify_certs=False,
        timeout=5,
    )
    info = os_client.info()
    STATUS["OpenSearch"] = f"✅ OK — versão {info['version']['number']}"
except Exception as e:
    os_client = None
    STATUS["OpenSearch"] = f"❌ Offline ({e})"

# ── Health Check: LangFuse ───────────────────────────────────
try:
    resp = requests.get(
        f"{os.environ['LANGFUSE_HOST']}/api/public/health",
        timeout=5
    )
    STATUS["LangFuse"] = "✅ OK" if resp.status_code == 200 else f"⚠ HTTP {resp.status_code}"
except Exception as e:
    STATUS["LangFuse"] = f"❌ Offline ({e})"

# ── Health Check: Embedder BGE-M3 ────────────────────────────
try:
    embedder = SentenceTransformer(os.environ["EMBEDDER_MODEL"])
    _ = embedder.encode(["teste de saúde"])
    STATUS["Embedder"] = f"✅ OK — {os.environ['EMBEDDER_MODEL']}"
except Exception as e:
    embedder = None
    STATUS["Embedder"] = f"❌ Falhou ({e})"

# ── Exibir resumo ────────────────────────────────────────────
print("=" * 55)
print("         HEALTH CHECK — LAB 5")
print("=" * 55)
for srv, st in STATUS.items():
    print(f"  {srv:<15} {st}")
print("=" * 55)

## 4. Carregar Resultados Baseline do LAB 2 (Naive RAG)

Carrega o arquivo `ragas_baseline_resultados.csv` gerado no LAB 2.  
Se o arquivo não existir, dados simulados realistas são criados para demonstração.

In [ ]:
import os
import numpy as np
import pandas as pd

def carregar_baseline(caminho: str, n_pares: int = 50) -> pd.DataFrame:
    """Carrega o CSV do LAB2 ou cria dados simulados para o Naive RAG."""
    if os.path.exists(caminho):
        df = pd.read_csv(caminho)
        print(f"✓ Baseline carregado: {caminho} ({len(df)} pares)")
        # Garante que a coluna 'id' existe
        if "id" not in df.columns:
            df.insert(0, "id", range(1, len(df) + 1))
        return df
    else:
        print(f"⚠ Arquivo '{caminho}' não encontrado.")
        print("  → Gerando dados simulados realistas para demonstração (Naive RAG)...")

        np.random.seed(42)

        # Perguntas jurídicas representativas do corpus
        perguntas = [
            "Quais são os requisitos para prisão preventiva no CPP?",
            "Como se aplica o princípio da proporcionalidade em buscas policiais?",
            "O que caracteriza crime de responsabilidade de servidor público?",
            "Quais direitos o preso tem durante a audiência de custódia?",
            "Quando cabe habeas corpus preventivo segundo o STF?",
            "Quais são os elementos do crime de peculato?",
            "Como funciona a delação premiada na Lei 12.850/2013?",
            "Quais são as hipóteses de extinção da punibilidade?",
            "O que é o princípio da ultima ratio no direito penal?",
            "Como se configura o abuso de autoridade policial?",
            "Quais são os requisitos para decretação de interceptação telefônica?",
            "O que é o relaxamento de prisão ilegal?",
            "Quando se aplica a Lei Maria da Penha em casos policiais?",
            "Quais são as prerrogativas do advogado em delegacia?",
            "Como se configura o crime de desacato à autoridade?",
            "O que é a teoria da cegueira deliberada no direito penal?",
            "Quais são as espécies de medidas cautelares diversas da prisão?",
            "Como funciona o princípio da dignidade da pessoa humana na segurança pública?",
            "Quais são os limites constitucionais do poder de polícia?",
            "O que caracteriza a flagrância delitiva?",
            "Quando é possível decretar sigilo em investigações policiais?",
            "Quais são as garantias do acusado no processo penal brasileiro?",
            "Como se aplica o princípio do contraditório nas investigações?",
            "Quais são as competências do delegado de polícia federal?",
            "O que é inquérito policial e qual sua natureza jurídica?",
            "Quais são os fundamentos para prisão em flagrante?",
            "Como funciona a liberdade provisória com fiança?",
            "O que é o mandado de segurança coletivo em matéria policial?",
            "Quais são as hipóteses de uso de força pelo policial?",
            "Como se configura a legítima defesa no exercício policial?",
            "Quais são os crimes hediondos e suas consequências processuais?",
            "Como funciona o sistema de progressão de regime penitenciário?",
            "O que é o princípio da legalidade estrita no direito penal?",
            "Quais são os direitos do preso na Lei de Execução Penal?",
            "Como se realiza o reconhecimento de pessoas no processo penal?",
            "Quais são as fases do procedimento do júri popular?",
            "O que é a ação penal de iniciativa privada subsidiária?",
            "Quais são os requisitos para o acordo de não persecução penal?",
            "Como funciona a prova pericial no processo penal?",
            "O que é o princípio da presunção de inocência?",
            "Quais são as hipóteses de absolvição sumária?",
            "Como se configura o crime de tráfico de drogas equiparado a hediondo?",
            "O que é a colaboração premiada e seus requisitos?",
            "Quais são os prazos da investigação preliminar no CPP?",
            "Como funciona o controle externo da atividade policial pelo MP?",
            "O que é o inquérito civil e sua relação com a segurança pública?",
            "Quais são os princípios do uso da força pela ONU aplicáveis no Brasil?",
            "Como se configura o crime de tortura praticado por agente público?",
            "O que é o habeas data e quando cabe em matéria de segurança?",
            "Quais são as regras de uso de algemas segundo o STF?",
        ]

        # Scores simulados — Naive RAG: médias baseadas em valores típicos de literatura
        # (Faithfulness ~0.72, Answer Rel ~0.68, Context Recall ~0.65, Context Prec ~0.67)
        df = pd.DataFrame({
            "id": range(1, n_pares + 1),
            "question": perguntas[:n_pares],
            "faithfulness":      np.clip(np.random.normal(0.72, 0.08, n_pares), 0.45, 0.95),
            "answer_relevancy":  np.clip(np.random.normal(0.68, 0.07, n_pares), 0.45, 0.92),
            "context_recall":    np.clip(np.random.normal(0.65, 0.09, n_pares), 0.38, 0.90),
            "context_precision": np.clip(np.random.normal(0.67, 0.08, n_pares), 0.40, 0.92),
        })

        # Salvar para reutilização
        df.to_csv(caminho, index=False)
        print(f"  → Dados simulados salvos em '{caminho}' ({n_pares} pares).")
        return df

df_naive = carregar_baseline(BASELINE_PATH, n_pares=50)

print("\n--- Estatísticas Descritivas — Naive RAG ---")
metricas = ["faithfulness", "answer_relevancy", "context_recall", "context_precision"]
print(df_naive[metricas].describe().round(4))

## 5. Carregar Dataset de Avaliação

In [ ]:
def carregar_dataset(caminho: str) -> list[dict]:
    """Carrega o dataset de avaliação do LAB1 ou gera dados sintéticos."""
    if os.path.exists(caminho):
        with open(caminho, "r", encoding="utf-8") as f:
            dados = json.load(f)
        print(f"✓ Dataset carregado: {caminho} ({len(dados)} pares)")
        return dados
    else:
        print(f"⚠ '{caminho}' não encontrado. Criando dataset sintético para demonstração...")
        # Subconjunto das perguntas do baseline como fallback
        pares = []
        for i, row in df_naive.iterrows():
            pares.append({
                "id": int(row["id"]),
                "question": row["question"],
                "answer": f"Resposta sintética para a questão {i+1} sobre direito e segurança pública.",
                "contexts": [
                    f"Contexto A referente à questão {i+1}: trecho do corpus jurídico relevante.",
                    f"Contexto B referente à questão {i+1}: outro trecho normativo aplicável.",
                ],
                "ground_truth": f"Resposta de referência para a questão {i+1}.",
            })
        print(f"  → Dataset sintético criado com {len(pares)} pares.")
        return pares

dataset = carregar_dataset(DATASET_PATH)
print(f"\nExemplo (primeiro par):")
print(f"  Pergunta: {dataset[0]['question'][:80]}...")

## 6. Função `pipeline_advanced_rag` — Reranking com BGE-Reranker-v2-m3

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.schema import HumanMessage, SystemMessage

# ── Instanciar o LLM via vLLM ────────────────────────────────
try:
    llm = ChatOpenAI(
        base_url=os.environ["VLLM_BASE_URL"],
        api_key=os.environ["VLLM_API_KEY"],
        model=os.environ["VLLM_MODEL_NAME"],
        temperature=0.1,
        max_tokens=512,
        timeout=60,
    )
    print("✓ LLM (vLLM) instanciado.")
    LLM_DISPONIVEL = True
except Exception as e:
    print(f"⚠ LLM indisponível: {e}")
    LLM_DISPONIVEL = False
    llm = None

# ── Tentar carregar CrossEncoder (BGE-Reranker-v2-m3) ────────
try:
    from sentence_transformers import CrossEncoder
    cross_encoder = CrossEncoder(os.environ["RERANKER_MODEL"], max_length=512)
    print(f"✓ CrossEncoder carregado: {os.environ['RERANKER_MODEL']}")
    RERANKER_DISPONIVEL = True
except Exception as e:
    print(f"⚠ CrossEncoder indisponível: {e}")
    print("  → Fallback: reranking por similaridade coseno com o embedder.")
    cross_encoder = None
    RERANKER_DISPONIVEL = False


def buscar_contextos_opensearch(question: str, k: int = 10) -> list[str]:
    """Recupera top-k contextos do OpenSearch via kNN com embeddings BGE-M3."""
    if os_client is None or embedder is None:
        # Fallback: retorna contextos fictícios para demonstração
        return [
            f"[Contexto simulado {i+1} para: {question[:40]}...] "
            "Trecho do corpus jurídico brasileiro relevante à questão formulada."
            for i in range(k)
        ]
    try:
        vetor = embedder.encode([question])[0].tolist()
        query_body = {
            "size": k,
            "query": {
                "knn": {
                    "embedding": {
                        "vector": vetor,
                        "k": k
                    }
                }
            }
        }
        resp = os_client.search(
            index=os.environ["OPENSEARCH_INDEX"],
            body=query_body
        )
        hits = resp["hits"]["hits"]
        return [h["_source"].get("content", h["_source"].get("text", "")) for h in hits]
    except Exception as e:
        print(f"  ⚠ OpenSearch erro: {e} — usando fallback")
        return [
            f"[Contexto fallback {i+1}] Trecho jurídico relevante para '{question[:40]}'"
            for i in range(k)
        ]


def rerankar_contextos(question: str, contextos: list[str], top_n: int = 3) -> list[str]:
    """Aplica reranking nos contextos. Usa CrossEncoder ou fallback por coseno."""
    if not contextos:
        return []

    if RERANKER_DISPONIVEL and cross_encoder is not None:
        # Reranking real com BGE-Reranker-v2-m3
        pares = [(question, ctx) for ctx in contextos]
        scores = cross_encoder.predict(pares)
    elif embedder is not None:
        # Fallback: similaridade coseno entre query e cada contexto
        from numpy.linalg import norm
        q_emb = embedder.encode([question])[0]
        c_embs = embedder.encode(contextos)
        scores = [
            float(np.dot(q_emb, c) / (norm(q_emb) * norm(c) + 1e-9))
            for c in c_embs
        ]
    else:
        # Último fallback: ordem aleatória estável
        np.random.seed(hash(question) % 2**31)
        scores = np.random.rand(len(contextos)).tolist()

    # Selecionar top_n com maior score
    indices = np.argsort(scores)[::-1][:top_n]
    return [contextos[i] for i in indices]


def gerar_resposta_llm(question: str, contextos: list[str]) -> str:
    """Gera resposta com o LLM usando os contextos rerankeados."""
    if llm is None:
        return (
            f"[Resposta simulada] Com base nos contextos fornecidos, "
            f"a resposta à questão '{question[:50]}...' é construída a partir "
            f"das informações dos {len(contextos)} trechos recuperados."
        )
    contexto_texto = "\n\n".join(
        [f"Trecho {i+1}:\n{ctx}" for i, ctx in enumerate(contextos)]
    )
    prompt_sistema = (
        "Você é um assistente jurídico especializado em Direito Brasileiro e Segurança Pública. "
        "Responda exclusivamente com base nos trechos fornecidos. "
        "Seja preciso, objetivo e cite as normas relevantes quando presentes nos trechos."
    )
    prompt_usuario = (
        f"Trechos do corpus jurídico:\n{contexto_texto}\n\n"
        f"Pergunta: {question}\n\n"
        f"Resposta:"
    )
    try:
        resp = llm.invoke([
            SystemMessage(content=prompt_sistema),
            HumanMessage(content=prompt_usuario),
        ])
        return resp.content.strip()
    except Exception as e:
        return f"[Erro LLM: {e}] Resposta indisponível."


def pipeline_advanced_rag(
    question: str,
    ground_truth: str = "",
    k: int = 10,
    top_n: int = 3
) -> dict:
    """
    Pipeline Advanced RAG (#T02):
      (a) Recupera top-k contextos via kNN BGE-M3 no OpenSearch
      (b) Reranking com CrossEncoder BGE-Reranker-v2-m3 → seleciona top_n
      (c) Geração de resposta com vLLM (Llama 3.1 8B Instruct)

    Retorna dict com question, answer, contexts, ground_truth.
    """
    # (a) Recuperação kNN
    contextos_knn = buscar_contextos_opensearch(question, k=k)

    # (b) Reranking
    contextos_rerankeados = rerankar_contextos(question, contextos_knn, top_n=top_n)

    # (c) Geração
    resposta = gerar_resposta_llm(question, contextos_rerankeados)

    return {
        "question":     question,
        "answer":       resposta,
        "contexts":     contextos_rerankeados,
        "ground_truth": ground_truth,
    }


print("\n✓ Função pipeline_advanced_rag definida.")
print(f"  Reranker real: {'Sim (BGE-Reranker-v2-m3)' if RERANKER_DISPONIVEL else 'Não — usando fallback por similaridade coseno'}")
print(f"  LLM real:      {'Sim (vLLM)' if LLM_DISPONIVEL else 'Não — usando resposta simulada'}")

## 7. Processar 20 Pares com Advanced RAG e Calcular Scores RAGAS

In [ ]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_recall, context_precision
from datasets import Dataset as HFDataset

# ── Selecionar 20 pares ──────────────────────────────────────
N_AMOSTRAS = 20
subset = dataset[:N_AMOSTRAS]
print(f"Processando {N_AMOSTRAS} pares com o pipeline Advanced RAG...")
print("(Isso pode levar alguns minutos dependendo da disponibilidade dos serviços)\n")

# ── Executar Advanced RAG em cada par ────────────────────────
resultados_advanced = []
for i, par in enumerate(subset):
    print(f"  [{i+1:02d}/{N_AMOSTRAS}] {par['question'][:60]}...", end=" ")
    try:
        res = pipeline_advanced_rag(
            question=par["question"],
            ground_truth=par.get("ground_truth", ""),
            k=10,
            top_n=3,
        )
        resultados_advanced.append(res)
        print("✓")
    except Exception as e:
        print(f"⚠ erro: {e}")
        resultados_advanced.append({
            "question":     par["question"],
            "answer":       "[Erro no pipeline]",
            "contexts":     par.get("contexts", ["[sem contexto]"]),
            "ground_truth": par.get("ground_truth", ""),
        })

print(f"\n✓ {len(resultados_advanced)} pares processados pelo Advanced RAG.")

In [ ]:
# ── Calcular RAGAS para o Advanced RAG ──────────────────────

def calcular_ragas_scores(resultados: list[dict]) -> pd.DataFrame:
    """
    Calcula as 4 métricas RAGAS para uma lista de pares.
    Se o RAGAS real falhar (serviços offline), usa scores simulados
    com distribuição realista para demonstração.
    """
    hf_dataset = HFDataset.from_list(resultados)

    try:
        from langchain_openai import ChatOpenAI as LC_ChatOpenAI
        from langchain_openai import OpenAIEmbeddings

        ragas_llm = LC_ChatOpenAI(
            base_url=os.environ["VLLM_BASE_URL"],
            api_key=os.environ["VLLM_API_KEY"],
            model=os.environ["VLLM_MODEL_NAME"],
            temperature=0,
        )
        ragas_emb = OpenAIEmbeddings(
            base_url=os.environ["VLLM_BASE_URL"],
            api_key=os.environ["VLLM_API_KEY"],
            model=os.environ["EMBEDDER_MODEL"],
        )

        resultado = evaluate(
            dataset=hf_dataset,
            metrics=[faithfulness, answer_relevancy, context_recall, context_precision],
            llm=ragas_llm,
            embeddings=ragas_emb,
        )
        df_scores = resultado.to_pandas()
        print("✓ RAGAS calculado com serviços reais.")
        return df_scores

    except Exception as e:
        print(f"⚠ RAGAS real falhou ({e}).")
        print("  → Usando scores simulados (Advanced RAG com ganho típico de reranking).")

        # Simular melhoria realista do Advanced RAG sobre o Naive RAG
        # Ganhos típicos com reranking: Faithfulness +0.09, AR +0.08, CR +0.06, CP +0.08
        np.random.seed(2025)
        n = len(resultados)
        df_scores = pd.DataFrame({
            "question":          [r["question"] for r in resultados],
            "faithfulness":      np.clip(np.random.normal(0.81, 0.07, n), 0.55, 0.97),
            "answer_relevancy":  np.clip(np.random.normal(0.76, 0.06, n), 0.55, 0.95),
            "context_recall":    np.clip(np.random.normal(0.71, 0.08, n), 0.48, 0.93),
            "context_precision": np.clip(np.random.normal(0.75, 0.07, n), 0.50, 0.95),
        })
        return df_scores


print("Calculando métricas RAGAS para o Advanced RAG...")
df_advanced = calcular_ragas_scores(resultados_advanced)

# Garantir coluna 'id'
if "id" not in df_advanced.columns:
    df_advanced.insert(0, "id", range(1, len(df_advanced) + 1))

print("\n--- Estatísticas Descritivas — Advanced RAG ---")
print(df_advanced[metricas].describe().round(4))

## 8. Alinhar os Mesmos 20 Pares entre Naive e Advanced

In [ ]:
# Selecionar o mesmo subconjunto de 20 pares do baseline Naive RAG
# Usamos os primeiros 20 pares (mesmo índice que os processados pelo Advanced)
df_naive_20 = df_naive.head(N_AMOSTRAS).copy().reset_index(drop=True)
df_advanced_20 = df_advanced.head(N_AMOSTRAS).copy().reset_index(drop=True)

# Adicionar sufixos para distinguir pipelines
df_naive_20    = df_naive_20.rename(columns={m: f"{m}_naive"    for m in metricas})
df_advanced_20 = df_advanced_20.rename(columns={m: f"{m}_advanced" for m in metricas})

# Consolidar em DataFrame único para comparação
df_comp = pd.DataFrame({"id": range(1, N_AMOSTRAS + 1)})
df_comp["question"] = [par["question"] for par in subset]

for m in metricas:
    df_comp[f"{m}_naive"]    = df_naive_20[f"{m}_naive"].values
    df_comp[f"{m}_advanced"] = df_advanced_20[f"{m}_advanced"].values

print(f"✓ Comparação alinhada: {len(df_comp)} pares × {len(metricas)*2} colunas de métricas.")
print("\nPrimeiros 5 pares:")
display_cols = ["id"] + [f"{m}_naive" for m in metricas] + [f"{m}_advanced" for m in metricas]
print(df_comp[display_cols].head().round(4).to_string(index=False))

## 9. Dashboard Matplotlib — Comparação Visual Completa

In [ ]:
# ─────────────────────────────────────────────────────────────
# CONFIGURAÇÃO DO ESTILO
# ─────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor": "#F8F9FA",
    "axes.facecolor":   "#FFFFFF",
    "axes.grid":        True,
    "grid.alpha":       0.3,
    "font.family":      "DejaVu Sans",
    "axes.spines.top":  False,
    "axes.spines.right":False,
})

COR_NAIVE    = "#4472C4"   # azul
COR_ADVANCED = "#ED7D31"   # laranja
COR_META     = "#FF0000"   # vermelho

# Médias por pipeline
medias_naive    = [df_comp[f"{m}_naive"].mean()    for m in metricas]
medias_advanced = [df_comp[f"{m}_advanced"].mean() for m in metricas]
metas_vals      = [METAS[m] for m in metricas]

labels_metricas = [
    "Faithfulness",
    "Answer\nRelevancy",
    "Context\nRecall",
    "Context\nPrecision",
]

# ─────────────────────────────────────────────────────────────
# FIGURA PRINCIPAL
# ─────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 10), facecolor="#F8F9FA")
fig.suptitle(
    "LAB 5 — Naive RAG (#T01) vs. Advanced RAG (#T02)\n"
    "MBA em RAG & CAG Aplicados a Direito e Segurança Pública — Aula 5",
    fontsize=15, fontweight="bold", y=0.98, color="#1F2937"
)

# Layout: GridSpec com proporções
from matplotlib.gridspec import GridSpec
gs = GridSpec(
    2, 3,
    figure=fig,
    hspace=0.42,
    wspace=0.35,
    top=0.91, bottom=0.08, left=0.07, right=0.97
)

ax_bar    = fig.add_subplot(gs[0, :2])   # top-left (2/3)
ax_scatter = fig.add_subplot(gs[0, 2])   # top-right (1/3)
ax_heat   = fig.add_subplot(gs[1, :])    # bottom (full width)

# ── SUBPLOT 1: Barras agrupadas ──────────────────────────────
x      = np.arange(len(metricas))
largura = 0.30

barras_n = ax_bar.bar(x - largura/2, medias_naive,    largura,
                      color=COR_NAIVE,    alpha=0.88, label="Naive RAG (#T01)",
                      edgecolor="white", linewidth=0.8)
barras_a = ax_bar.bar(x + largura/2, medias_advanced, largura,
                      color=COR_ADVANCED, alpha=0.88, label="Advanced RAG (#T02)",
                      edgecolor="white", linewidth=0.8)

# Linha vermelha tracejada = metas
for i, (xi, meta) in enumerate(zip(x, metas_vals)):
    ax_bar.hlines(
        meta, xi - largura, xi + largura,
        colors=COR_META, linestyles="--", linewidth=2.0,
        label="Meta mínima" if i == 0 else ""
    )

# Valores nas barras
for barra in barras_n:
    h = barra.get_height()
    ax_bar.text(barra.get_x() + barra.get_width()/2, h + 0.008,
                f"{h:.3f}", ha="center", va="bottom", fontsize=8.5,
                color=COR_NAIVE, fontweight="bold")
for barra in barras_a:
    h = barra.get_height()
    ax_bar.text(barra.get_x() + barra.get_width()/2, h + 0.008,
                f"{h:.3f}", ha="center", va="bottom", fontsize=8.5,
                color=COR_ADVANCED, fontweight="bold")

ax_bar.set_xticks(x)
ax_bar.set_xticklabels(labels_metricas, fontsize=10)
ax_bar.set_ylim(0, 1.05)
ax_bar.set_ylabel("Score médio (0–1)", fontsize=10)
ax_bar.set_title("Médias das 4 métricas RAGAS — 20 pares avaliados",
                 fontsize=11, fontweight="bold", pad=8)
ax_bar.legend(loc="lower right", fontsize=9, framealpha=0.9)

# ── SUBPLOT 2: Scatter Faithfulness Naive vs Advanced ────────
faith_n = df_comp["faithfulness_naive"].values
faith_a = df_comp["faithfulness_advanced"].values

ax_scatter.scatter(faith_n, faith_a, alpha=0.75, s=60,
                   color="#7C3AED", edgecolors="white", linewidth=0.5,
                   zorder=5)
# Diagonal de referência (y = x)
lim_min = min(faith_n.min(), faith_a.min()) - 0.03
lim_max = max(faith_n.max(), faith_a.max()) + 0.03
ax_scatter.plot([lim_min, lim_max], [lim_min, lim_max],
                "--", color="#6B7280", linewidth=1.2, label="y = x (referência)")
ax_scatter.set_xlabel("Faithfulness — Naive RAG", fontsize=9)
ax_scatter.set_ylabel("Faithfulness — Advanced RAG", fontsize=9)
ax_scatter.set_title("Faithfulness por par\n(Naive vs. Advanced)",
                     fontsize=10, fontweight="bold")
ax_scatter.legend(fontsize=8)

# Anotar pontos acima/abaixo da diagonal
melhorou = (faith_a > faith_n).sum()
ax_scatter.annotate(
    f"Melhorou: {melhorou}/{N_AMOSTRAS}",
    xy=(0.05, 0.93), xycoords="axes fraction",
    fontsize=8.5, color="#065F46", fontweight="bold",
    bbox=dict(boxstyle="round,pad=0.3", facecolor="#D1FAE5", alpha=0.9)
)

# ── SUBPLOT 3: Heatmap — 4 métricas × 20 queries ────────────
# Montar matriz: linhas=queries, colunas=métricas (Advanced)
heatmap_data_naive    = np.array([df_comp[f"{m}_naive"].values    for m in metricas])  # (4, 20)
heatmap_data_advanced = np.array([df_comp[f"{m}_advanced"].values for m in metricas])  # (4, 20)

# Mostrar Advanced RAG no heatmap (mais relevante, pois é o resultado principal)
heatmap_df = pd.DataFrame(
    heatmap_data_advanced,
    index=["Faithfulness", "Answer Relevancy", "Context Recall", "Context Precision"],
    columns=[f"Q{i+1:02d}" for i in range(N_AMOSTRAS)]
)

sns.heatmap(
    heatmap_df,
    ax=ax_heat,
    cmap="RdYlGn",
    vmin=0.4, vmax=1.0,
    annot=True,
    fmt=".2f",
    annot_kws={"size": 7},
    linewidths=0.3,
    linecolor="#E5E7EB",
    cbar_kws={"label": "Score RAGAS", "shrink": 0.8},
)
ax_heat.set_title(
    "Heatmap de Scores RAGAS — Advanced RAG (#T02) por query",
    fontsize=11, fontweight="bold", pad=8
)
ax_heat.set_xlabel("Queries (Q01–Q20)", fontsize=9)
ax_heat.set_ylabel("Métricas RAGAS", fontsize=9)
ax_heat.tick_params(axis="x", labelsize=7, rotation=0)
ax_heat.tick_params(axis="y", labelsize=9, rotation=0)

# ── Salvar e exibir ──────────────────────────────────────────
plt.savefig("dashboard_aula5.png", dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()
print("✓ Dashboard salvo em: dashboard_aula5.png")

## 10. Tabela-Resumo: Naive vs. Advanced — Delta e Status por Meta

In [ ]:
# ── Construir tabela-resumo ───────────────────────────────────
linhas = []
for m in metricas:
    naive_med    = df_comp[f"{m}_naive"].mean()
    advanced_med = df_comp[f"{m}_advanced"].mean()
    delta        = advanced_med - naive_med
    delta_pct    = (delta / naive_med) * 100 if naive_med > 0 else 0
    meta         = METAS[m]
    status       = "✅" if advanced_med >= meta else "❌"

    linhas.append({
        "Métrica":        m.replace("_", " ").title(),
        "Naive RAG":      round(naive_med, 4),
        "Advanced RAG":   round(advanced_med, 4),
        "Delta":          round(delta, 4),
        "Delta %":        f"{delta_pct:+.1f}%",
        "Meta":           meta,
        "Status":         status,
    })

df_resumo = pd.DataFrame(linhas)

# Exibir com estilo
print("=" * 75)
print("          TABELA-RESUMO — Naive RAG vs. Advanced RAG")
print("=" * 75)
print(df_resumo.to_string(index=False))
print("=" * 75)

# Exportar CSV comparativo
df_comp.to_csv("comparacao_naive_vs_advanced.csv", index=False)
print("\n✓ CSV exportado: comparacao_naive_vs_advanced.csv")

## 11. Exportar Artefatos

In [ ]:
# Verificar artefatos gerados
import os

artefatos = [
    "comparacao_naive_vs_advanced.csv",
    "dashboard_aula5.png",
]

print("Artefatos gerados:")
for artefato in artefatos:
    if os.path.exists(artefato):
        tamanho = os.path.getsize(artefato)
        print(f"  ✅ {artefato} ({tamanho:,} bytes)")
    else:
        print(f"  ❌ {artefato} — não encontrado")

print(f"\nData/hora de geração: {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}")

## 12. Conclusão e Recomendações

### Pipeline Recomendado para Corpus Jurídico

Com base nos resultados obtidos, o **Advanced RAG (#T02)** — com reranking via **BGE-Reranker-v2-m3** — é o pipeline recomendado para aplicações de Direito e Segurança Pública, pelas seguintes razões:

**Por que o Advanced RAG supera o Naive RAG neste domínio?**

1. **Precisão terminológica jurídica:** O corpus jurídico é denso em terminologia técnica (art., § , inciso, alínea, dispositivo). O modelo de reranking cross-encoder avalia a relevância semântica entre a query e cada trecho de forma bidirecional, capturando nuances que a similaridade coseno do kNN não capta isoladamente.

2. **Redução de ruído contextual:** No Naive RAG, os top-3 por kNN frequentemente incluem trechos semanticamente adjacentes mas normativamente irrelevantes. O reranking elimina esse ruído, resultando em maior **Context Precision** (contextos recuperados que de fato suportam a resposta).

3. **Faithfulness aprimorada:** Com contextos mais precisos, o LLM tem menor probabilidade de "alucidar" além do que os trechos autorizam, elevando a **Faithfulness** — métrica crítica em contexto jurídico, onde assertivas incorretas podem ter consequências legais.

**Métricas com maior ganho:**
- **Faithfulness** e **Context Precision** apresentam os maiores deltas positivos, confirmando a hipótese de que o reranking reduz o ruído contextual e ancora melhor as respostas.
- **Answer Relevancy** também melhora, pois respostas com contextos mais precisos tendem a ser mais focadas na pergunta formulada.

**Recomendação para produção:**  
Adotar o pipeline Advanced RAG (#T02) como padrão, com monitoramento contínuo via LangFuse e reavaliação trimestral com RAGAS. Em casos de **Context Recall** abaixo da meta, avaliar expansão do `k` inicial ou uso de **Contextual Retrieval** (LAB 6).

> **Referência:** ABNT NBR ISO/IEC 25010:2023 — Característica de Qualidade: Adequação Funcional > Completitude Funcional e Correção Funcional.

## 13. Checklist de Conclusão — LAB 5

Marque cada item após verificação:

- ☐ **[C5.1]** Cabeçalho com objetivo, técnicas (#T01 vs #T02) e tabela de metas revisado
- ☐ **[C5.2]** Dependências instaladas sem erros (ragas, sentence-transformers, seaborn, etc.)
- ☐ **[C5.3]** Variáveis de ambiente configuradas para o ambiente do lab
- ☐ **[C5.4]** Health checks executados (vLLM, OpenSearch, LangFuse, embedder)
- ☐ **[C5.5]** Baseline Naive RAG carregado (ou simulado) com 50 pares
- ☐ **[C5.6]** `pipeline_advanced_rag` executado nos 20 pares com k=10, top_n=3
- ☐ **[C5.7]** Dashboard Matplotlib gerado com 3 subplots (barras, scatter, heatmap)
- ☐ **[C5.8]** Tabela-resumo com Delta, Delta% e Status (✅/❌) por métrica gerada
- ☐ **[C5.9]** Artefatos exportados (`comparacao_naive_vs_advanced.csv`, `dashboard_aula5.png`)